In [1]:
import os
import json
from dotenv import load_dotenv
from pydantic import (BaseModel, Field, ValidationError)
from typing import (List, Literal)
from azure.identity import (DefaultAzureCredential, get_bearer_token_provider)
from azure.storage.blob import BlobServiceClient
from langchain_openai import (ChatOpenAI, OpenAIEmbeddings)
from langgraph.graph import (StateGraph, START, END)
from langchain.agents import create_agent

In [2]:
load_dotenv()

True

In [3]:
azure_project_endpoint = os.getenv("AZURE_PROJECT_ENDPOINT")
azure_model_endpoint = os.getenv("AZURE_MODEL_ENDPOINT")
azure_model_name = os.getenv("AZURE_MODEL_NAME")
azure_embedding_model_name = os.getenv("AZURE_EMBEDDING_MODEL_NAME")

In [4]:
credential = DefaultAzureCredential()

token_provider = get_bearer_token_provider(
    credential,
    "https://ai.azure.com/.default",
)

In [5]:
llm = ChatOpenAI(
    model=azure_model_name,
    base_url=f"{azure_project_endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
    use_responses_api=True,
)

In [6]:
def load_employee_data():
    """Download employee JSON data from Azure Blob Storage."""
    account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
    container_name = os.environ["AZURE_STORAGE_CONTAINER_EMPLOYEES"]
    blob_name = os.environ["AZURE_STORAGE_BLOB_NAME_EMPLOYEES"]

    credential = DefaultAzureCredential()

    try:
        service = BlobServiceClient(
            account_url=account_url,
            credential=credential,
        )

        blob = service.get_blob_client(
            container=container_name,
            blob=blob_name,
        )

        json_bytes = blob.download_blob().readall()
        data = json.loads(json_bytes.decode("utf-8"))

        # Accept either a list or {"employees": [...]}.
        if isinstance(data, dict):
            data = data.get("employees")

        if not isinstance(data, list):
            raise ValueError(
                "Employee data must be a JSON list or an "
                "object containing an 'employees' list."
            )

        return data

    finally:
        credential.close()

In [7]:
def load_task_data():
    """Download task JSON data from Azure Blob Storage."""

    account_url = os.environ["AZURE_STORAGE_ACCOUNT_URL"]
    container_name = os.environ["AZURE_STORAGE_CONTAINER_TASKS"]
    blob_name = os.environ["AZURE_STORAGE_BLOB_NAME_TASKS"]
    
    credential = DefaultAzureCredential()

    try:
        service = BlobServiceClient(
            account_url=account_url,
            credential=credential,
        )

        blob = service.get_blob_client(
            container=container_name,
            blob=blob_name,
        )

        json_bytes = blob.download_blob().readall()
        data = json.loads(json_bytes.decode("utf-8"))

        # Accept either a list or {"tasks": [...]}.
        if isinstance(data, dict):
            data = data.get("tasks")

        if not isinstance(data, list):
            raise ValueError(
                "Task data must be a JSON list or an "
                "object containing a 'tasks' list."
            )

        return data

    finally:
        credential.close()

In [8]:
employee_data = load_employee_data()
employee_data

[{'id': 0,
  'name': 'Jonas Greslund',
  'position': 'Software Engineer',
  'skill_map': {'angular': 'expert',
   'dotnet': 'advanced',
   'python': 'advanced',
   'sql': 'advanced',
   'pulumi': 'novice'}},
 {'id': 1,
  'name': 'Link',
  'position': 'Software Engineer',
  'skill_map': {'angular': 'advanced',
   'dotnet': 'intermediate',
   'python': 'advanced',
   'sql': 'intermediate',
   'pulumi': 'intermediate'}},
 {'id': 2,
  'name': 'Zeenula Pelphegopoulos',
  'position': 'Senior Software Engineer',
  'skill_map': {'angular': 'intermediate',
   'dotnet': 'advanced',
   'python': 'expert',
   'sql': 'advanced',
   'pulumi': 'expert'}},
 {'id': 3,
  'name': 'Peach',
  'position': 'Software Quality Engineer',
  'skill_map': {'selenium': 'advanced', 'regression_testing': 'advanced'}}]

In [9]:
task_data = load_task_data()
task_data

[{'id': 0,
  'title': 'Handle the edge case with the UI of the scoring performance chart.',
  'description': 'There are times when 2 different z-scores have a value of zero.',
  'required_skills': ['Angular', 'css']},
 {'id': 1,
  'title': 'Deploy infrastructure',
  'description': 'Provision cloud resources using Pulumi.',
  'required_skills': ['pulumi', 'python']},
 {'id': 2,
  'title': 'Optimize database queries',
  'description': 'Improve the performance of database queries to reduce latency.',
  'required_skills': ['sql', 'python']},
 {'id': 3,
  'title': 'Conduct regression testing',
  'description': 'Perform regression testing to ensure that new code changes do not break existing functionality.',
  'required_skills': ['selenium', 'regression_testing']}]

In [10]:
def evaluate_employee_skill_set(employee_name, skill, level, evalation_agent):
    """
    This function returns an evaluation of a particular employee's skills.
    """

    evaluation_prompt = f"""
        The employee's name is {employee_name}.
        The skill under evaluation is {skill}.
        The level of expertise is {level}.
    """

    result = evaluation_agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ]
    })
    return result

In [11]:
evaluation_agent_system_prompt = (
    "You are an employee skill evaluator. "
    "Your job is to read an employee's skill as well as how advanced they are at that particular skill. "
    "You must output a brief statement about their level at that skill."
)

evaluation_agent = create_agent(
    model=llm,
    tools=[], # We can add tools later.
    system_prompt=evaluation_agent_system_prompt
)

In [12]:
class EmployeeEvaluation(BaseModel):
    employee_id: int
    evaluation: str

In [15]:
evaluations: List[EmployeeEvaluation] = []
for employee in employee_data:
    for skill, level in employee['skill_map'].items():
        evaluation = evaluate_employee_skill_set(employee['name'], skill, level, evaluation_agent)
        evaluation_content = evaluation['messages'][1].content
        evaluation_text = evaluation_content[0]['text']
        evaluations.append({
            "employee_id": employee["id"],
            "evaluation": evaluation_text
        })

In [13]:
# Initialize with an embedding model
embeddings_model = OpenAIEmbeddings(
    model=azure_embedding_model_name,
    base_url=f"{azure_model_endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)

In [16]:
evaluations

[{'employee_id': 0,
  'evaluation': 'Jonas Greslund demonstrates expert-level proficiency in Angular, with advanced knowledge of the framework and the ability to design, develop, and optimize complex applications.'},
 {'employee_id': 0,
  'evaluation': 'Jonas Greslund demonstrates advanced proficiency in .NET, with strong expertise in developing and maintaining applications using the framework.'},
 {'employee_id': 0,
  'evaluation': 'Jonas Greslund demonstrates advanced proficiency in Python, with strong expertise in developing complex applications and solving challenging programming problems.'},
 {'employee_id': 0,
  'evaluation': 'Jonas Greslund demonstrates advanced SQL proficiency, with strong capabilities in complex queries, data manipulation, and query optimization.'},
 {'employee_id': 0,
  'evaluation': 'Jonas Greslund is a novice in Pulumi, with a foundational understanding and a need for guidance when defining and managing infrastructure as code.'},
 {'employee_id': 1,
  'eval

In [17]:
class TaskEmployeeCandidate(BaseModel):
    employee_id: int
    task_id: int
    learning_opportunity: Literal["Low", "Medium", "High"] = Field(
        description="Employee's learning opportunity. Low means limited opportunity; High means significant opportunity."
    )

    leadership_capability: Literal["Low", "Medium", "High"] = Field(
        description="Employee's technical leadership and mentoring capability."
    )

    learning_explanation: str = Field(
        description="Brief explanation supporting the learning opportunity classification."
    )

    leadership_explanation: str = Field(
        description="Brief explanation supporting the leadership capability classification."
    )

In [18]:
skill_matching_agent_system_prompt = (
    "You are a technical skill evaluation and employee matching agent. "

    "YOUR OBJECTIVE: "
    "Evaluate an employee's technical skills and proficiency using "
    "a list of required skills and the available employee evaluation. "
    "Your goal is to identify both the employee's potential for learning "
    "and their ability to provide technical leadership to others. "

    "SKILL EVALUATION: "
    "Evaluate each listed skill individually. "
    "Use the provided evaluation as the primary source of evidence. "
    "You may infer transferable knowledge when the employee has experience "
    "with related technologies or concepts. For example, Angular experience "
    "may indicate familiarity with CSS because Angular applications commonly "
    "use CSS for styling. However, related experience does not automatically "
    "prove proficiency in the associated skill. "
    "Distinguish between demonstrated proficiency and inferred proficiency. "
    "Never invent experience or assume expertise without supporting evidence. "

    "If a listed skill is not mentioned in the evaluation and there is "
    "insufficient evidence to infer proficiency, treat that skill as unknown. "
    "Do not automatically classify unknown skills as beginner-level skills. "

    "LEARNING OPPORTUNITY: "
    "Classify the employee's opportunity to learn and develop the required "
    "technical skills. "
    "A higher classification indicates greater learning opportunities. "
    "A lower classification indicates that the employee already demonstrates "
    "strong proficiency in the required skills. "

    "Use these classification guidelines: "
    "LOW: The employee demonstrates advanced or expert proficiency in nearly "
    "all required skills and has limited opportunities to learn. "
    "MEDIUM: The employee demonstrates a mixture of proficiency levels "
    "or has some meaningful opportunities to develop. "
    "HIGH: The employee demonstrates beginner-level proficiency in several "
    "required skills and has substantial opportunities to learn and develop. "

    "LEADERSHIP CAPABILITY: "
    "Classify the employee's potential to provide technical leadership, "
    "mentorship, or guidance to other employees in the required skills. "
    "A higher classification indicates stronger demonstrated technical "
    "expertise and greater potential to mentor others. "
    "A lower classification indicates limited demonstrated proficiency. "

    "Use these classification guidelines: "
    "LOW: The employee demonstrates beginner-level proficiency in most "
    "required skills and has limited ability to provide technical guidance. "
    "MEDIUM: The employee demonstrates intermediate proficiency or has "
    "advanced proficiency in some required skills and may be able to "
    "provide guidance in those areas. "
    "HIGH: The employee demonstrates advanced or expert proficiency across "
    "most required skills and has strong potential to provide technical "
    "leadership and mentorship. "

    "SCORING CONSIDERATIONS: "
    "Evaluate learning opportunity and leadership capability independently. "
    "Do not assume that a HIGH learning opportunity classification "
    "automatically requires a LOW leadership classification, or vice versa. "
    "Consider the number of required skills, the employee's demonstrated "
    "proficiency, and the relevance of transferable experience. "
    "If the available evidence is insufficient to confidently determine "
    "either classification, use MEDIUM as the neutral classification. "
    "Do not interpret missing skill information as proof of beginner "
    "or expert proficiency. "

    "STRICT CLASSIFICATION REQUIREMENTS: "
    "For both learning_opportunity and leadership_capability, you MUST "
    "return exactly one of the following values: "
    "'Low', 'Medium', or 'High'. "
    "Do not return numerical scores, percentages, or alternative labels. "
    "Use the exact capitalization shown. "

    "OUTPUT REQUIREMENTS: "
    "Return both classifications along with a brief explanation supporting "
    "each classification. "
    "Explanations must reference the employee's actual skills and "
    "evaluation evidence. "
    "Do not invent qualifications, experience, or technical expertise. "
)

skill_matching_agent = create_agent(
    model=llm,
    tools=[], # We can add tools later.
    system_prompt=evaluation_agent_system_prompt,
    response_format=TaskEmployeeCandidate
)

In [25]:
def match_task_to_skills(task_id, skills, evaluation, skill_matching_agent):
    print(evaluation)

    skill_matching_prompt = f"""
        The required skills are: {skills}
        The employee evaluation is: {evaluation['evaluation']}
        The employee id is: {evaluation['employee_id']}
    """

    try:
        result = skill_matching_agent.invoke({
            "messages": [
                {"role": "user", "content": skill_matching_prompt}
            ]
        })

        return result["structured_response"]

    except (ValidationError, Exception) as e:
        print(f"Skill matching agent failed: {e}")

        return TaskEmployeeCandidate(
            employee_id=evaluation['employee_id'],
            task_id=task_id,
            learning_opportunity="Medium",
            leadership_capability="Medium",
            learning_explanation="Unable to determine learning opportunity.",
            leadership_explanation="Unable to determine leadership capability."
        )

In [26]:
skill_matches = []
for task in task_data:
    skills = ", ".join(task['required_skills'])
    for evaluation in evaluations:
        skill_match = match_task_to_skills(task['id'], skills, evaluation, skill_matching_agent)
        skill_matches.append(skill_match)

{'employee_id': 0, 'evaluation': 'Jonas Greslund demonstrates expert-level proficiency in Angular, with advanced knowledge of the framework and the ability to design, develop, and optimize complex applications.'}
Skill matching agent failed: Failed to parse structured output for tool 'TaskEmployeeCandidate': Failed to parse data to TaskEmployeeCandidate: 1 validation error for TaskEmployeeCandidate
task_id
  Field required [type=missing, input_value={'employee_id': 0, 'learn...ce are not documented.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing.
{'employee_id': 0, 'evaluation': 'Jonas Greslund demonstrates advanced proficiency in .NET, with strong expertise in developing and maintaining applications using the framework.'}
Skill matching agent failed: Failed to parse structured output for tool 'TaskEmployeeCandidate': Failed to parse data to TaskEmployeeCandidate: 1 validation error for TaskEmployeeCandidate
task_id
  Field required [ty

In [27]:
for skill_match in skill_matches:
    print(skill_match)
    print('-')

employee_id=0 task_id=0 learning_opportunity='Medium' leadership_capability='Medium' learning_explanation='Unable to determine learning opportunity.' leadership_explanation='Unable to determine leadership capability.'
-
employee_id=0 task_id=0 learning_opportunity='Medium' leadership_capability='Medium' learning_explanation='Unable to determine learning opportunity.' leadership_explanation='Unable to determine leadership capability.'
-
employee_id=0 task_id=0 learning_opportunity='Medium' leadership_capability='Medium' learning_explanation='Unable to determine learning opportunity.' leadership_explanation='Unable to determine leadership capability.'
-
employee_id=0 task_id=0 learning_opportunity='Medium' leadership_capability='Medium' learning_explanation='Unable to determine learning opportunity.' leadership_explanation='Unable to determine leadership capability.'
-
employee_id=0 task_id=0 learning_opportunity='Medium' leadership_capability='Medium' learning_explanation='Unable to det